In [ ]:
import pandas as pd
import duckdb
import duckdb
import os
import glob

In [ ]:
def carregar_csv_pandas_para_duckdb(pasta_input: str,
                                    caminho_banco: str,
                                    delimitador: str = ";"):
    """
    Versão simples: lê CSVs usando pandas (ISO-8859-1) e cria tabelas no schema 'raw'.
    O nome da tabela é baseado no nome do arquivo (sem extensão).
    """

    # Conecta ao banco
    con = duckdb.connect(caminho_banco)

    # Garante schema raw
    con.execute("CREATE SCHEMA IF NOT EXISTS raw;")

    # Encontrar todos os CSV
    arquivos = glob.glob(os.path.join(pasta_input, "*.csv"))

    if not arquivos:
        print(" Nenhum CSV encontrado na pasta.")
        return

    for caminho in arquivos:
        nome_arquivo = os.path.basename(caminho)
        nome_tabela = os.path.splitext(nome_arquivo)[0].lower().replace(" ", "_")
        tabela_schema = f"raw.{nome_tabela}"

        print(f"Lendo CSV (ISO-8859-1): {nome_arquivo}")

        # Lendo CSV com pandas
        df = pd.read_csv(
            caminho,
            sep=delimitador,
            encoding="ISO-8859-1",
            dtype=str   # garante leitura estável
        )

        # Registrar o DataFrame no DuckDB
        con.register("tmp_df", df)

        # Criar tabela definitiva
        con.execute(f"CREATE TABLE {tabela_schema} AS SELECT * FROM tmp_df;")

        print(f"Tabela criada: {tabela_schema} ({len(df)} linhas)")

    con.close()
    print(" Carga concluída com sucesso!")

In [ ]:
# Caminho do arquivo (pode ser CSV, Parquet, etc)
PATH_DIRETORIO = "data_input"

# Caminho do banco DuckDB (arquivo .db)
CAMINHO_DUCKDB = "bd/dev.duckdb"

# Cria a conexão com o banco DuckDB
con = duckdb.connect(CAMINHO_DUCKDB)


In [ ]:
carregar_csv_pandas_para_duckdb(pasta_input=PATH_DIRETORIO, caminho_banco=CAMINHO_DUCKDB, delimitador=";")

In [ ]:
con.close()